# Resume Keyword Analyzer — NLP Pipeline

**Author:** Astha Raghav | B.Tech CSE (IoT), ABES Institute of Technology

---

## What This Does
Compares a resume against a job description using **TF-IDF keyword extraction**  
and returns a **match score, matched keywords, missing keywords, and improvement tips**.

## Pipeline
```
Resume / JD text
   → Clean (remove noise, URLs, emails, special chars)
   → Tokenise (split, remove stopwords)
   → TF-IDF extraction (top keywords per document)
   → Match keywords (intersection)
   → Score = matched / total JD keywords × 100
   → Report
```

## Key Result
Reduces manual resume screening from **15–20 minutes → under 5 seconds** (240× speedup)  
Handles real-world edge cases: noisy text, encoding issues, missing sections.

## Step 1: Install Libraries

In [ ]:
# Install libraries not available in Colab by default
!pip install PyPDF2 python-docx -q

import re
import string
import time
import json
import os
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import warnings
warnings.filterwarnings('ignore')

os.makedirs('/content/results', exist_ok=True)

print('All imports successful')

## Step 2: Text Cleaner Module

Handles noise removal from raw text: URLs, emails, phone numbers, special characters, encoding errors.  
This is the first stage of the pipeline — garbage in, garbage out.

In [ ]:
STOPWORDS = {
    'a','an','the','and','or','but','in','on','at','to','for','of','with',
    'by','from','as','is','are','was','were','be','been','being','have',
    'has','had','do','does','did','will','would','could','should','may',
    'might','shall','can','i','me','my','we','our','you','your','he','him',
    'his','she','her','it','its','they','them','their','this','that','these',
    'those','am','into','about','also','any','both','each','few','more',
    'most','no','not','only','such','too','very','just','now','so','then',
    'when','where','while','who','why','how','what','which','there','here',
    'up','out','use','used','make','like','even','still','never','ever'
}


def clean_text(text):
    """
    Clean raw text string.
    Removes: URLs, emails, phone numbers, special characters, extra whitespace.
    Returns: cleaned, lowercased string.
    """
    if not text or not isinstance(text, str):
        return ''

    # Fix encoding issues
    text = text.encode('utf-8', errors='ignore').decode('utf-8', errors='ignore')

    text = re.sub(r'http\S+|www\S+', ' ', text)          # remove URLs
    text = re.sub(r'\S+@\S+', ' ', text)                  # remove emails
    text = re.sub(r'[\+\(]?[1-9][0-9 .\-\(\)]{8,}[0-9]', ' ', text)  # remove phones
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)           # keep only alphanumeric
    text = re.sub(r'\s+', ' ', text)                       # collapse whitespace
    return text.strip().lower()


# --- TEST IT ---
test_inputs = [
    'Contact: astha@email.com | Phone: +91 9876543210',
    'Visit https://github.com/user for my projects!',
    'Skills: Python • Flask • SQL | REST API & Machine Learning',
    'B.Tech CSE (IoT) — 82% | ABES Institute of Technology',
]

print('--- Cleaner Test ---')
for t in test_inputs:
    print(f'Input : {t}')
    print(f'Output: {clean_text(t)}')
    print()

## Step 3: TF-IDF Keyword Extractor

**TF-IDF** = Term Frequency × Inverse Document Frequency.  
Words that appear often in this document but rarely in general English score highest.  
We also extract **bigrams** (e.g. 'machine learning', 'data analysis') for better matching.

In [ ]:
def extract_keywords(text, top_n=30):
    """
    Extract top N keywords using TF-IDF.
    Returns: list of (keyword, score) sorted by score descending.
    """
    if not text or len(text.split()) < 5:
        return []

    vectorizer = TfidfVectorizer(
        max_features=500,
        ngram_range=(1, 2),       # unigrams + bigrams
        stop_words='english',
        min_df=1,
        sublinear_tf=True         # log normalisation
    )

    try:
        matrix = vectorizer.fit_transform([text])
        feature_names = vectorizer.get_feature_names_out()
        scores = matrix.toarray()[0]
        pairs = [(w, round(float(s), 4)) for w, s in zip(feature_names, scores) if s > 0]
        pairs.sort(key=lambda x: x[1], reverse=True)
        return pairs[:top_n]
    except Exception as e:
        print(f'TF-IDF failed: {e}')
        return []


# --- TEST IT ---
sample = 'python developer flask rest api machine learning pandas numpy sql database git github agile sprint'
keywords = extract_keywords(sample, top_n=15)
print('--- Top Keywords ---')
for kw, score in keywords:
    print(f'  {kw:<30} {score:.4f}')

## Step 4: Match Scorer

Compares resume keywords against JD keywords.  
**Score** = (keywords found in both) / (total JD keywords) × 100

In [ ]:
def calculate_match_score(resume_text, jd_text, top_n=40):
    """
    Compare resume against job description.
    Returns dict with score, matched keywords, missing keywords, verdict.
    """
    if not resume_text or not jd_text:
        return {'score': 0, 'matched': [], 'missing': [], 'verdict': 'Error'}

    jd_kws     = extract_keywords(jd_text,     top_n=top_n)
    resume_kws = extract_keywords(resume_text, top_n=top_n * 2)

    jd_set     = {kw for kw, _ in jd_kws}
    resume_set = {kw for kw, _ in resume_kws}

    matched = sorted(jd_set & resume_set)
    missing = sorted(jd_set - resume_set)

    score = round(len(matched) / len(jd_set) * 100, 1) if jd_set else 0

    if score >= 70:   verdict = 'Strong Match'
    elif score >= 40: verdict = 'Partial Match'
    else:             verdict = 'Weak Match'

    return {
        'score':           score,
        'verdict':         verdict,
        'matched':         matched,
        'missing':         missing,
        'total_jd_kws':    len(jd_set),
        'total_matched':   len(matched),
        'resume_keywords': [kw for kw, _ in resume_kws[:20]],
        'jd_keywords':     [kw for kw, _ in jd_kws[:20]],
    }


# --- TEST WITH SAMPLE DATA ---
sample_resume = '''
Astha Raghav — Python Developer
Skills: Python, Flask, REST API, MySQL, Pandas, NumPy, Scikit-learn, NLP, Git, GitHub
Projects: Resume Analyzer (Python, NLP, TF-IDF, Flask API)
          Task Management App (Node.js, React, MySQL)
          Crop Yield Prediction (Scikit-learn, ML, Pandas)
Internship: Edunet Foundation — Python backend modules, ML pipelines, debugging
Education: B.Tech CSE IoT 82% ABES Institute of Technology
'''

sample_jd = '''
We are looking for a Python developer with experience in Flask or Django,
REST API design, SQL databases, machine learning, and data analysis.
Must know Git, have strong problem solving skills, and be comfortable
working with pandas, numpy, and scikit-learn. Experience with NLP is a plus.
'''

result = calculate_match_score(clean_text(sample_resume), clean_text(sample_jd))

print('=' * 50)
print(f'  Score   : {result["score"]}%')
print(f'  Verdict : {result["verdict"]}')
print(f'  Matched : {result["total_matched"]} / {result["total_jd_kws"]} JD keywords')
print('=' * 50)
print(f'  Matched keywords : {result["matched"]}')
print(f'  Missing keywords : {result["missing"]}')

## Step 5: Speed Benchmark

Key claim on resume: **15–20 minutes (manual) → under 5 seconds (automated)**.  
Let's prove it by timing the full pipeline.

In [ ]:
import time

# Simulate a realistic resume (longer text)
realistic_resume = '''
Astha Raghav
B.Tech Computer Science Engineering IoT 82 percent
ABES Institute of Technology Ghaziabad

SKILLS
Python OOP functions file io error handling debugging
SQL MySQL joins group by subqueries indexing
Pandas NumPy data cleaning transformations missing values
Flask REST API design consumption JSON Postman
React js Node js HTML CSS Bootstrap
Scikit-learn classification regression cross validation feature engineering
NLP TF-IDF text processing tokenisation
Git GitHub branching pull requests structured commits
Linux CLI VS Code Agile Sprint workflows
Hugging Face LangChain basic awareness

EXPERIENCE
Software Development Intern Edunet Foundation June 2025 August 2025
Built 3 Python backend modules for ML data pipelines
Resolved 5 critical bugs in live pipeline scripts
Refactored slow Pandas data loading module
Authored module documentation I O contracts error codes

PROJECTS
Resume Keyword Analyzer Python Flask NLP TF-IDF REST API Pandas
Task Management Web Application React Node MySQL REST API Git
Crop Yield Prediction Python Scikit-learn Pandas NumPy
Django URL Shortener Django MySQL REST API Python

EDUCATION
Machine Learning for All Coursera
Java Foundation Infosys Springboard
Project Management Udemy
Research paper Scopus indexed IMACSI 2025 predictive analytics smart monitoring
'''

realistic_jd = '''
Python developer intern required with experience in REST API design,
machine learning, data analysis, Flask or Django framework,
SQL database management, Git version control, agile workflows.
Knowledge of Pandas, NumPy, Scikit-learn preferred.
Good communication skills and ability to work in team environment.
Experience with NLP or data pipelines is a bonus.
'''

# Time the full pipeline
N_RUNS = 20
times = []

for i in range(N_RUNS):
    start = time.time()
    r_clean = clean_text(realistic_resume)
    j_clean = clean_text(realistic_jd)
    result  = calculate_match_score(r_clean, j_clean)
    elapsed = time.time() - start
    times.append(elapsed)

avg_time = sum(times) / len(times)
max_time = max(times)

print('=' * 50)
print('  SPEED BENCHMARK')
print('=' * 50)
print(f'  Runs              : {N_RUNS}')
print(f'  Average time      : {avg_time:.4f} seconds')
print(f'  Max time          : {max_time:.4f} seconds')
print(f'  Manual equivalent : 15-20 minutes = 900-1200 seconds')
print(f'  Speedup factor    : ~{int(900 / avg_time)}x faster')
print('=' * 50)
print(f'\nFinal result from last run:')
print(f'  Score  : {result["score"]}%')
print(f'  Verdict: {result["verdict"]}')

## Step 6: Edge Case Handling

Real-world documents are messy. The pipeline must handle:  
malformed input, empty documents, encoding errors, and very short text.

In [ ]:
print('--- Edge Case Tests ---\n')

# Test 1: Empty inputs
r = calculate_match_score('', 'python developer flask')
assert r['score'] == 0, 'FAIL: empty input should return 0'
print('PASS: empty resume returns score=0')

# Test 2: Very short text
r = clean_text('hi')
assert len(r.split()) <= 1
print('PASS: very short text handled')

# Test 3: Special characters and unicode
messy = 'Python • Flask | SQL — REST API & ML / NLP'
r = clean_text(messy)
assert 'python' in r and 'flask' in r and 'sql' in r
print('PASS: special characters cleaned correctly')

# Test 4: Email and URL removal
with_noise = 'Contact me at user@email.com or visit https://github.com/user'
r = clean_text(with_noise)
assert '@' not in r and 'http' not in r
print('PASS: email and URLs removed')

# Test 5: Encoding error
with_encoding = 'caf\u00e9 r\u00e9sum\u00e9 na\u00efve'
r = clean_text(with_encoding)
assert isinstance(r, str)
print('PASS: encoding errors handled gracefully')

# Test 6: Partial match scoring
resume = 'python flask sql git machine learning'
jd     = 'python django sql postgresql docker machine learning'
r = calculate_match_score(resume, jd)
assert 0 < r['score'] < 100
assert len(r['matched']) > 0
assert len(r['missing']) > 0
print(f'PASS: partial match = {r["score"]}% ({len(r["matched"])} matched, {len(r["missing"])} missing)')

# Test 7: Matched keywords not in missing
overlap = set(r['matched']) & set(r['missing'])
assert len(overlap) == 0
print('PASS: matched and missing are mutually exclusive')

print('\nAll edge case tests passed.')

## Step 7: Visualise Results

In [ ]:
# Use the realistic resume + JD from the benchmark cell
r_clean = clean_text(realistic_resume)
j_clean = clean_text(realistic_jd)
result  = calculate_match_score(r_clean, j_clean)

# Chart 1: Match score gauge
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Pie chart showing matched vs missing
matched_count = result['total_matched']
missing_count = result['total_jd_kws'] - matched_count
axes[0].pie(
    [matched_count, missing_count],
    labels=[f'Matched ({matched_count})', f'Missing ({missing_count})'],
    colors=['#27AE60', '#E74C3C'],
    autopct='%1.1f%%',
    startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
axes[0].set_title(f'Keyword Match: {result["score"]}% ({result["verdict"]})',
                  fontweight='bold', fontsize=13)

# Bar chart: top resume keywords by TF-IDF score
top_kws = extract_keywords(r_clean, top_n=12)
kw_names = [k for k, _ in top_kws]
kw_scores = [s for _, s in top_kws]
colors_bar = ['#27AE60' if k in set(result['matched']) else '#4A90D9' for k in kw_names]

axes[1].barh(kw_names, kw_scores, color=colors_bar, edgecolor='white')
axes[1].set_xlabel('TF-IDF Score')
axes[1].set_title('Top Resume Keywords\n(green = matched with JD)', fontweight='bold')
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig('/content/results/keyword_match.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: /content/results/keyword_match.png')

In [ ]:
# Chart 2: Speed comparison
categories   = ['Manual\nScreening', 'This Tool']
times_sec    = [1050, avg_time]  # 17.5 min avg manual = 1050 sec
display_vals = ['~17 min', f'{avg_time:.3f} sec']
bar_colors   = ['#E74C3C', '#27AE60']

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(categories, [17.5 * 60, avg_time * 1000],
              color=bar_colors, width=0.4, edgecolor='white')
ax.set_ylabel('Time (milliseconds, log scale)')
ax.set_yscale('log')
ax.set_title('Manual Screening vs Automated Pipeline', fontweight='bold', fontsize=13)
for bar, label in zip(bars, display_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.2,
            label, ha='center', fontweight='bold', fontsize=12)
plt.tight_layout()
plt.savefig('/content/results/speed_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Speedup: ~{int(1050 / avg_time)}x faster than manual')

## Step 8: Complete Pipeline Function

A single function that runs the entire pipeline — clean → extract → score → report.

In [ ]:
def analyse_resume(resume_text, jd_text, verbose=True):
    """
    Full pipeline: clean → extract keywords → score → report.

    Parameters:
        resume_text : raw resume string (paste from any format)
        jd_text     : raw job description string
        verbose     : print formatted report (default True)

    Returns:
        dict with score, verdict, matched, missing, tips
    """
    start = time.time()

    # Stage 1: Clean
    resume_clean = clean_text(resume_text)
    jd_clean     = clean_text(jd_text)

    # Stage 2: Score
    result = calculate_match_score(resume_clean, jd_clean)

    # Stage 3: Tips
    tips = []
    for kw in result['missing'][:8]:
        tips.append(f"Add '{kw}' to your resume if you have this skill")
    result['tips']     = tips
    result['time_sec'] = round(time.time() - start, 4)

    if verbose:
        print('=' * 55)
        print('  RESUME ANALYSIS REPORT')
        print('=' * 55)
        print(f'  Match Score : {result["score"]}%')
        print(f'  Verdict     : {result["verdict"]}')
        print(f'  Matched     : {result["total_matched"]} / {result["total_jd_kws"]} JD keywords')
        print(f'  Time taken  : {result["time_sec"]} seconds')
        print()
        print(f'  MATCHED  : {", ".join(result["matched"][:10])}')
        print(f'  MISSING  : {", ".join(result["missing"][:10])}')
        print()
        if tips:
            print('  TIPS TO IMPROVE YOUR SCORE:')
            for tip in tips:
                print(f'    - {tip}')
        print('=' * 55)

    return result


# Run it!
final = analyse_resume(realistic_resume, realistic_jd)

## Step 9: Try With Your Own Resume & JD

Paste your resume text and job description below and run the cell.

In [ ]:
# ============================================================
# PASTE YOUR RESUME TEXT HERE
# ============================================================
MY_RESUME = """
Paste your resume text here...
"""

# ============================================================
# PASTE THE JOB DESCRIPTION HERE
# ============================================================
MY_JD = """
Paste the job description here...
"""

# Run analysis
if len(MY_RESUME.strip()) > 50 and len(MY_JD.strip()) > 20:
    my_result = analyse_resume(MY_RESUME, MY_JD)
else:
    print('Please paste your resume and JD text above, then re-run this cell.')

## Step 10: Download Results

In [ ]:
from google.colab import files
import zipfile

# Save analysis results as JSON
with open('/content/results/analysis_result.json', 'w') as f:
    json.dump(final, f, indent=2)

# Zip everything
with zipfile.ZipFile('/content/resume_analyzer_results.zip', 'w') as zf:
    for fname in os.listdir('/content/results'):
        zf.write(f'/content/results/{fname}', f'results/{fname}')

print('Files in /content/results:')
for f in sorted(os.listdir('/content/results')):
    size = os.path.getsize(f'/content/results/{f}') / 1024
    print(f'  {f:<40} {size:.1f} KB')

print('\nDownloading...')
files.download('/content/resume_analyzer_results.zip')

## Summary

### What this notebook demonstrates

| Step | What | Skill demonstrated |
|---|---|---|
| Text cleaning | Remove noise, URLs, encoding issues | Data preprocessing |
| TF-IDF extraction | Weighted keyword importance | NLP / feature engineering |
| Keyword matching | Set intersection scoring | Algorithm design |
| Edge cases | Empty input, encoding errors, short text | Robust engineering |
| Speed benchmark | 240x faster than manual | Performance measurement |

### Key metric
Manual resume screening: **15–20 minutes**  
This pipeline: **under 5 seconds** — a **240x speedup**

### Next steps
- Wrap in Flask REST API (see `run.py` in the repo)
- Add PDF/DOCX upload support
- Build a simple web UI
- Deploy on Render or Railway

---
**Author:** Astha Raghav | [LinkedIn](https://linkedin.com/in/astha-raghav)